In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType,DateType,BooleanType
import pyspark.sql.functions as f
catalog_name='ecommerce'

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_order_items")
display(df_bronze)

In [0]:
df_silver = df_bronze.dropDuplicates(["order_id","item_seq"])
df_silver= df_silver.withColumn("quantity",f.when(f.col("quantity")=="Two",2).otherwise(f.col("quantity").cast("int")))\
    .withColumn("unit_price",f.regexp_replace(f.col("unit_price"),"[$]","").cast("double"))\
    .withColumn("discount_pct",f.regexp_replace(f.col("discount_pct"),"%","").cast("double"))\
    .withColumn("coupon_code",f.lower(f.trim(f.col("coupon_code"))))\
    .withColumn("channel",f.when(f.col("channel") == "web", "Website").when(f.col("channel") == "app", "Mobile").otherwise(f.col("channel")))\
    .withColumn("dt",f.to_date(f.col("dt"),"yyyy-MM-dd"))\
    .withColumn("order_ts",f.coalesce(f.to_timestamp("order_ts", "yyyy-MM-dd HH:mm:ss"), f.to_timestamp("order_ts", "dd-MM-yyyy HH:mm")))\
    .withColumn("item_seq",f.col("item_seq").cast("int"))\
    .withColumn("tax_amount",f.regexp_replace(f.col("tax_amount"), r"[^0-9.\-]", "").cast("double"))\
    .withColumn("processed_time", F.current_timestamp())

display(df_silver)


In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_order_items")